# open-leaks: Pipeline Prototyping

Run individual pipeline stages manually for testing and development.

```bash
pip install -e libs/dagster-io -e packages/open-leaks
```

In [ ]:
import os
# os.environ["LLM_BASE_URL"] = "http://litellm.your-cluster:4000/v1"
# os.environ["LLM_API_KEY"] = "sk-..."

## 1. Create Sample Leak Documents

In [ ]:
from open_leaks.core.document import Document

sample_docs = [
    Document(
        id="wikileaks-cable-001",
        title="Embassy Cable: Political Situation Report",
        content=(
            "SUBJECT: Political developments in Country X\n\n"
            "1. (C) Summary: Recent elections have shifted the political landscape significantly. "
            "The ruling party lost 15 seats in parliament. Opposition leader John Smith has "
            "called for early elections. The Ambassador met with Foreign Minister Jane Doe "
            "on March 5, 2009 to discuss bilateral relations. The Ministry of Finance reported "
            "GDP growth of 3.2% for Q4 2008.\n\n"
            "2. (C) The European Union expressed concern over press freedom restrictions. "
            "Reporters Without Borders downgraded the country's press freedom index. "
            "The United Nations Human Rights Council scheduled a review for June 2009."
        ),
        source="wikileaks",
        document_type="cable",
        domain="open_leaks",
        entity_type="Cable",
        metadata={"classification": "CONFIDENTIAL", "origin": "Embassy X"},
    ),
    Document(
        id="icij-panama-entity-001",
        title="Offshore Holdings Ltd",
        content=(
            "Offshore Holdings Ltd is a shell company incorporated in the British Virgin Islands "
            "on January 15, 2005. The registered agent is Mossack Fonseca. Directors include "
            "nominee director services provided by Acme Corporate Services Ltd. The company "
            "holds bank accounts at Credit Suisse in Zurich and HSBC in Hong Kong. "
            "Annual filings show transfers to accounts in Panama and Luxembourg."
        ),
        source="icij-panama-papers",
        document_type="offshore_entity",
        domain="open_leaks",
        entity_type="OffshoreEntity",
        metadata={"jurisdiction": "BVI", "dataset": "panama-papers"},
    ),
]

print(f"Created {len(sample_docs)} sample documents")
for doc in sample_docs:
    print(f"  {doc.id}: {doc.title} ({len(doc.content)} chars)")

## 2. Chunking

In [ ]:
from dagster_io import chunk_document

all_chunks = []
for doc in sample_docs:
    chunks = chunk_document(
        document_id=doc.id,
        title=doc.title,
        content=doc.content,
        metadata={"source": doc.source, "document_type": doc.document_type},
        chunk_size=300,
        chunk_overlap=50,
    )
    all_chunks.extend(chunks)

print(f"{len(sample_docs)} docs → {len(all_chunks)} chunks")
for chunk in all_chunks:
    print(f"\n{chunk.chunk_id} ({len(chunk.text)} chars):")
    print(f"  {chunk.text[:100]}...")

## 3. NER Extraction

In [ ]:
from dagster_io import LLMResource
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

llm = LLMResource()
llm.setup_for_execution(None)

class Entity(BaseModel):
    text: str = Field(description="Entity mention")
    label: str = Field(description="PERSON, ORG, GPE, DATE, LAW, EVENT")
    context: str = Field(description="Sentence fragment")

class NERResult(BaseModel):
    entities: list[Entity]

chain = llm.with_structured_output(NERResult)

for chunk in all_chunks[:2]:  # first 2 chunks
    result = chain.invoke([
        SystemMessage(content="Extract all named entities from this leaked document."),
        HumanMessage(content=chunk.text),
    ])
    print(f"\n--- {chunk.chunk_id} ---")
    for ent in result.entities:
        print(f"  [{ent.label}] {ent.text}")

## 4. S-P-O Proposition Extraction

In [ ]:
class Proposition(BaseModel):
    subject: str
    predicate: str
    object: str
    confidence: float = Field(ge=0, le=1)

class PropositionResult(BaseModel):
    propositions: list[Proposition]

spo_chain = llm.with_structured_output(PropositionResult)

chunk = all_chunks[0]
result = spo_chain.invoke([
    SystemMessage(content="Extract S-P-O triples. Focus on factual claims."),
    HumanMessage(content=chunk.text),
])

print(f"Propositions from {chunk.chunk_id}:")
for p in result.propositions:
    print(f"  ({p.subject}) --[{p.predicate}]--> ({p.object})  [{p.confidence:.0%}]")

## 5. Embedding & Similarity

In [ ]:
from dagster_io import EmbeddingResource
import numpy as np

emb = EmbeddingResource()
emb.setup_for_execution(None)

texts = [c.text for c in all_chunks]
vectors = emb.embed(texts)

print(f"Embedded {len(vectors)} chunks → {len(vectors[0])}d")

# Query similarity
query = "offshore shell companies and bank accounts"
query_vec = emb.embed_single(query)

sims = [
    np.dot(query_vec, v) / (np.linalg.norm(query_vec) * np.linalg.norm(v))
    for v in vectors
]

print(f"\nQuery: '{query}'")
for chunk, sim in sorted(zip(all_chunks, sims), key=lambda x: -x[1]):
    print(f"  {sim:.3f} | {chunk.chunk_id}: {chunk.text[:60]}...")